# Analisis Sentimen Komentar TikTok Progres IKN Menggunakan Neural Network
## Perbandingan Model ANN, SimpleRNN, LSTM, GRU, dan BiLSTM

### Identitas
- **Nama**: Asyam Sakhi Maulana
- **NIM**: 240535600363
- **Offering**: Teknik Informatika A
- **Mata Kuliah**: Pemrosesan Bahasa Alami (NLP) - Pertemuan 7

---
### Deskripsi Tugas
Pada praktikum ini, dilakukan analisis sentimen terhadap dataset komentar TikTok mengenai progres pembangunan Ibu Kota Nusantara (IKN) — dataset yang sama persis dengan praktikum pertemuan 6. Jika pada pertemuan 6 pendekatan yang digunakan adalah **Classical Machine Learning** (TF-IDF + Naive Bayes, Logistic Regression, dan SVM), pada praktikum ini pendekatan ditingkatkan menggunakan arsitektur **Deep Learning / Neural Network**.

Seluruh arsitektur neural network utama untuk pemrosesan teks dieksplorasi dan dibandingkan:
1. **ANN (Artificial Neural Network / Dense Feedforward)**
2. **SimpleRNN (Recurrent Neural Network)**
3. **LSTM (Long Short-Term Memory)**
4. **GRU (Gated Recurrent Unit)**
5. **BiLSTM (Bidirectional Long Short-Term Memory)**

Setiap model diuji melalui proses:
- Preprocessing teks khusus bahasa Indonesia dengan mempertahankan kata negasi penting.
- Tokenisasi sequence dan padding untuk representasi deep learning.
- Pelatihan dengan **Early Stopping** untuk mencegah overfitting.
- Uji **variasi hyperparameter** (Baseline vs Tuned: variasi units, dropout, embedding dimension).
- Evaluasi komprehensif berdasarkan **Confusion Matrix**: Accuracy, Precision, Recall, dan F1-Score.


## 1. Instalasi dan Import Library

Library yang digunakan meliputi:
- **pandas** & **numpy**: Manipulasi data tabel dan array numerik.
- **matplotlib** & **seaborn**: Visualisasi distribusi data, kurva loss/akurasi pelatihan, serta confusion matrix.
- **Sastrawi**: Preprocessing teks bahasa Indonesia (stopword removal dan stemming).
- **scikit-learn**: Pembagian dataset (train-test split berstrata) serta metrik evaluasi (accuracy, precision, recall, f1-score, confusion matrix).
- **tensorflow / keras**: Pembangunan dan pelatihan arsitektur neural network (Embedding, Dense, SimpleRNN, LSTM, GRU, Bidirectional, Dropout, EarlyStopping).


In [ ]:
!pip install --quiet Sastrawi tensorflow scikit-learn seaborn matplotlib pandas numpy

import os
import re
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Deep Learning / TensorFlow
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Dense, Dropout, GlobalAveragePooling1D,
    SimpleRNN, LSTM, GRU, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Set random seed untuk reproduktibilitas eksperimen
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print("Semua library berhasil dimuat!")


## 2. Pemuatan dan Eksplorasi Dataset (EDA)

Dataset yang digunakan adalah **Data Komentar Tiktok Progres IKN** (Kaggle: `najmarafifah/analisis-sentimen-komentar-tiktok-progres-ikn`), dataset yang sama dengan Pertemuan 6. Dataset ini berisi opini masyarakat berbahasa Indonesia informal mengenai pembangunan IKN dengan label sentimen biner: **positif** dan **negatif**.

Kode di bawah ini secara otomatis mendeteksi apakah file dataset sudah berada di folder kerja (misal file `.xlsx` atau `.csv`). Jika belum ada dan dijalankan di Google Colab, kode akan menampilkan dialog upload file.


In [ ]:
# Cek ketersediaan file dataset di direktori lokal atau Colab
data_files = glob.glob("*.xlsx") + glob.glob("*.csv")

if data_files:
    dataset_path = data_files[0]
    print(f"File dataset ditemukan: {dataset_path}")
    if dataset_path.endswith('.xlsx'):
        df = pd.read_excel(dataset_path)
    else:
        df = pd.read_csv(dataset_path)
else:
    try:
        from google.colab import files
        print("Silakan upload file dataset sentimen IKN (.xlsx atau .csv)...")
        uploaded = files.upload()
        dataset_path = list(uploaded.keys())[0]
        if dataset_path.endswith('.xlsx'):
            df = pd.read_excel(dataset_path)
        else:
            df = pd.read_csv(dataset_path)
    except ImportError:
        # Fallback dataset dummy untuk pengujian lokal jika file fisik belum dipindahkan
        print("Peringatan: Modul google.colab tidak ditemukan dan file dataset tidak ada di folder.")
        print("Membuat sampel data kompatibel untuk verifikasi alur kerja...")
        df = pd.DataFrame({
            "text": [
                "semoga proyek ikn ini berjalan lancar dan sukses untuk indonesia",
                "hambur hamburkan uang rakyat saja proyek tidak jelas",
                "bangga banget melihat pembangunan nusantara semakin megah",
                "mending dananya buat perbaiki jalan di daerah jangan dipaksakan",
                "semangat para pekerja ikn karya anak bangsa luar biasa",
                "ujung ujungnya mangkrak kayak proyek jaman dulu pesimis saya"
            ] * 250,
            "label": ["positif", "negatif", "positif", "negatif", "positif", "negatif"] * 250
        })

print("\n5 Baris Teratas Dataset:")
display(df.head())


### Pemeriksaan Struktur dan Pembersihan Duplikat
Dilakukan pemeriksaan dimensi, tipe data, missing values, dan data duplikat agar kualitas dataset terjamin sebelum masuk ke tahap pelatihan model.


In [ ]:
print("Informasi Dimensi dan Kolom Dataset:")
print(f"Dimensi awal: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Nama kolom  : {df.columns.tolist()}\n")

print("Pemeriksaan Missing Values:")
print(df.isnull().sum())
print()

# Identifikasi dan hapus data duplikat
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat ditemukan: {jumlah_duplikat}")

if jumlah_duplikat > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dimensi setelah menghapus duplikat: {df.shape[0]} baris")
else:
    print("Tidak ditemukan baris duplikat.")


### Distribusi Kelas Sentimen
Memeriksa keseimbangan antara komentar berlabel `positif` dan `negatif`. Keseimbangan kelas penting untuk menentukan apakah metrik Accuracy dapat diandalkan atau memerlukan penyesuaian bobot kelas.


In [ ]:
distribusi = df['label'].value_counts()
persentase = df['label'].value_counts(normalize=True) * 100

summary_label = pd.DataFrame({
    'Jumlah Sampel': distribusi,
    'Persentase (%)': persentase.round(2)
})
print("Distribusi Frekuensi Kelas:")
display(summary_label)

# Visualisasi distribusi label
plt.figure(figsize=(7, 4))
colors = ['#e74c3c', '#2ecc71']
ax = sns.barplot(x=distribusi.index, y=distribusi.values, palette=colors)
plt.title("Distribusi Label Sentimen Komentar Progres IKN", fontsize=13, fontweight='bold')
plt.xlabel("Sentimen", fontsize=11)
plt.ylabel("Jumlah Komentar", fontsize=11)

for p in ax.patches:
    ax.annotate(f"{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


## 3. Preprocessing Teks Khusus Bahasa Indonesia

Proses pembersihan teks mengikuti standar pipeline NLP bahasa Indonesia dengan tahapan:
1. **Cleaning**: Menghapus URL, mention (`@user`), angka, karakter non-huruf, tanda baca, dan spasi berlebih menggunakan regular expression.
2. **Case Folding**: Mengubah seluruh teks menjadi huruf kecil (*lowercase*) agar representasi kata konsisten.
3. **Stopword Removal**: Menghapus kata umum yang tidak membawa sentimen menggunakan daftar stopword **Sastrawi**.
   > **Catatan Krusial**: Kata negasi seperti `tidak`, `bukan`, `jangan`, `nggak`, `gak`, `ga`, `tak`, dan `kagak` **dikecualikan dari stopword removal**. Kata-kata ini sangat menentukan pembalik polaritas sentimen (misal: "tidak bagus" vs "bagus").
4. **Stemming**: Memotong imbuhan kata dasar bahasa Indonesia menggunakan algoritma **Sastrawi** (misal: "membangun" -> "bangun", "kemajuan" -> "maju").


In [ ]:
# Definisi himpunan kata negasi yang wajib dipertahankan
negasi = {"tidak", "bukan", "jangan", "nggak", "gak", "ga", "tak", "kagak"}

# Siapkan stopword Sastrawi dan kecualikan kata negasi
stopword_factory = StopWordRemoverFactory()
daftar_stopword_dasar = set(stopword_factory.get_stop_words())
daftar_stopword_final = daftar_stopword_dasar - negasi

# Siapkan stemmer Sastrawi
stemmer = StemmerFactory().create_stemmer()

def cleaning(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # Hapus URL
    text = re.sub(r"@\w+", " ", text)                # Hapus mention @user
    text = re.sub(r"\d+", " ", text)                 # Hapus angka
    text = re.sub(r"[^a-zA-Z\s]", " ", text)         # Hapus tanda baca & simbol
    text = re.sub(r"\s+", " ", text).strip()         # Normalisasi spasi
    return text

def preprocess_text(text):
    # 1. Cleaning
    text = cleaning(text)
    # 2. Case Folding
    text = text.lower()
    # 3. Tokenisasi dasar
    tokens = text.split()
    # 4. Stopword Removal (kata negasi dipertahankan)
    tokens = [t for t in tokens if t not in daftar_stopword_final]
    # 5. Stemming Sastrawi
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

# Demonstrasi tahapan pada satu contoh kalimat
contoh = "Proyek IKN @jokowi ini JANGAN sampai mangkrak ya!! Harus sukses 100% http://tiktok.com/ikn"
print("--- Contoh Uji Coba Fungsi Preprocessing ---")
print("Teks Asli  :", contoh)
print("Hasil Bersih:", preprocess_text(contoh))


### Penerapan Preprocessing ke Seluruh Dataset
Menerapkan fungsi `preprocess_text` pada seluruh kolom komentar dan membuang baris yang menjadi kosong setelah preprocessing.


In [ ]:
print("Menjalankan preprocessing pada seluruh dataset (mohon tunggu sebentar)...")
df['text_bersih'] = df['text'].apply(preprocess_text)

# Hapus baris kosong setelah preprocessing
df = df[df['text_bersih'].str.strip() != ""].reset_index(drop=True)

print(f"Total data siap pakai setelah preprocessing: {len(df)} baris\n")
print("Contoh Perbandingan 5 Data Pertama (Sebelum vs Sesudah Preprocessing):")
display(df[['text', 'text_bersih', 'label']].head())


## 4. Tokenisasi, Sequencing, dan Padding untuk Neural Network

### Perbedaan Representasi Teks: Classical NLP vs Deep Learning
- **Classical NLP (TF-IDF)**: Merepresentasikan teks sebagai matriks bobot frekuensi kata (*Bag-of-Words*). Urutan kata dan konteks sekuensial hilang.
- **Deep Learning (Word Embedding + Sequence)**: Setiap kata dipetakan menjadi token integer unik. Urutan waktu kata (*word sequence*) dipertahankan sehingga model recurrent (RNN/LSTM/GRU) dapat mempelajari dependensi kata dari awal hingga akhir kalimat.

### Langkah-langkah:
1. **Label Encoding**: Mengubah label kategori (`negatif` -> 0, `positif` -> 1).
2. **Train/Test Split**: Pembagian 80% data latih dan 20% data uji dengan `stratify=y` dan `random_state=42` (sama persis dengan baseline Pertemuan 6).
3. **Keras Tokenizer**: Fit kamus kata hanya pada data latih (`X_train`) untuk mencegah **data leakage**.
4. **Sequencing & Padding**: Menyeragamkan panjang kalimat dengan menambahkan nol (*padding*) hingga panjang maksimum (`max_len`).


In [ ]:
# 1. Label Encoding (0 = negatif, 1 = positif)
label_mapping = {'negatif': 0, 'positif': 1}
df['label_num'] = df['label'].map(label_mapping)

# 2. Train-Test Split (80:20 stratified)
X = df['text_bersih'].values
y = df['label_num'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Jumlah Data Latih (Train): {len(X_train_raw)} sampel")
print(f"Jumlah Data Uji (Test)   : {len(X_test_raw)} sampel")

# 3. Analisis Panjang Kalimat untuk Menentukan max_len yang Optimal
panjang_kata = [len(teks.split()) for teks in X_train_raw]
q95_len = int(np.percentile(panjang_kata, 95))
max_len = max(50, q95_len)

print(f"Panjang kata rata-rata : {np.mean(panjang_kata):.1f}")
print(f"Panjang kata persentil 95: {q95_len}")
print(f"Max Length yang dipilih: {max_len} token")

# Visualisasi distribusi panjang kalimat
plt.figure(figsize=(7, 3.5))
sns.histplot(panjang_kata, bins=25, kde=True, color='purple')
plt.axvline(max_len, color='red', linestyle='--', label=f'max_len = {max_len}')
plt.title("Distribusi Panjang Kata Komentar (Data Latih)")
plt.xlabel("Jumlah Kata")
plt.ylabel("Frekuensi")
plt.legend()
plt.tight_layout()
plt.show()

# 4. Tokenisasi menggunakan Keras Tokenizer
VOCAB_SIZE = 5000  # Maksimum kosakata paling sering muncul
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_raw)

actual_vocab_size = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)
print(f"Jumlah kosakata unik pada data latih: {len(tokenizer.word_index)}")
print(f"Ukuran vocabulary yang digunakan    : {actual_vocab_size}")

# 5. Konversi teks ke sequence integer & padding
X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_test_seq = tokenizer.texts_to_sequences(X_test_raw)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print(f"Bentuk tensor data latih (X_train_pad): {X_train_pad.shape}")
print(f"Bentuk tensor data uji   (X_test_pad) : {X_test_pad.shape}")


## 5. Arsitektur Model Neural Network

Kita membangun dan mengomparasikan **5 arsitektur neural network** untuk klasifikasi sentimen teks:

| No | Model | Karakteristik Utama | Kelebihan & Limitasi |
|:---|:---|:---|:---|
| 1 | **ANN (Dense / FFNN)** | *Feedforward Neural Network* dengan *GlobalAveragePooling1D* setelah Embedding | Cepat dilatih, namun mengabaikan struktur urutan kata (*bag-of-embeddings*). |
| 2 | **SimpleRNN** | Jaringan *Recurrent* dasar dengan loop umpan balik tersembunyi | Mampu menangkap konteks sekuensial pendek, rentan masalah *vanishing/exploding gradient*. |
| 3 | **LSTM** | *Long Short-Term Memory* dengan mekanisme 3 gate (Forget, Input, Output) dan *Cell State* | Mampu mengingat informasi jangka panjang (*long-term dependencies*) dan mengatasi *vanishing gradient*. |
| 4 | **GRU** | *Gated Recurrent Unit* dengan 2 gate (Reset & Update Gate) tanpa *Cell State* terpisah | Lebih ringkas dan cepat dilatih dibanding LSTM dengan performa komparabel pada teks pendek. |
| 5 | **BiLSTM** | *Bidirectional LSTM* yang membaca sekuens teks dari arah maju dan mundur sekaligus | Menangkap konteks kata dari masa lalu (kata sebelum) dan masa depan (kata sesudah) secara simultan. |

Semua model menggunakan layer **Embedding** di awal dan layer output `Dense(1, activation='sigmoid')` dengan fungsi loss `binary_crossentropy`.


In [ ]:
def create_neural_model(model_type, vocab_size, embedding_dim, units, dropout_rate, max_length):
    """
    Fungsi modular untuk membangun model Neural Network berdasarkan tipe arsitektur.
    """
    model = Sequential(name=f"Model_{model_type}")
    
    # Layer 1: Word Embedding
    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_length,
        name="embedding_layer"
    ))
    
    # Layer 2: Core Processing Layer
    if model_type == "ANN":
        # Global Average Pooling merata-ratakan vektor representasi kata
        model.add(GlobalAveragePooling1D(name="global_avg_pool"))
        model.add(Dense(units, activation='relu', name="dense_hidden_1"))
        model.add(Dropout(dropout_rate, name="dropout_1"))
        
    elif model_type == "SimpleRNN":
        model.add(SimpleRNN(units, return_sequences=False, name="simple_rnn"))
        model.add(Dropout(dropout_rate, name="dropout_1"))
        model.add(Dense(units // 2, activation='relu', name="dense_hidden_1"))
        
    elif model_type == "LSTM":
        model.add(LSTM(units, return_sequences=False, name="lstm"))
        model.add(Dropout(dropout_rate, name="dropout_1"))
        model.add(Dense(units // 2, activation='relu', name="dense_hidden_1"))
        
    elif model_type == "GRU":
        model.add(GRU(units, return_sequences=False, name="gru"))
        model.add(Dropout(dropout_rate, name="dropout_1"))
        model.add(Dense(units // 2, activation='relu', name="dense_hidden_1"))
        
    elif model_type == "BiLSTM":
        model.add(Bidirectional(LSTM(units, return_sequences=False), name="bilstm"))
        model.add(Dropout(dropout_rate, name="dropout_1"))
        model.add(Dense(units // 2, activation='relu', name="dense_hidden_1"))
        
    else:
        raise ValueError(f"Tipe model {model_type} tidak dikenali.")
        
    # Layer Output: Klasifikasi Biner (Sigmoid)
    model.add(Dense(1, activation='sigmoid', name="output_layer"))
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Contoh inspeksi ringkasan arsitektur BiLSTM
contoh_model = create_neural_model("BiLSTM", actual_vocab_size, 64, 64, 0.2, max_len)
contoh_model.summary()


## 6. Pelatihan Model & Eksperimen Tuning Hyperparameter

Sesuai instruksi praktikum, dilakukan minimal 1 variasi eksperimen pada parameter `units`, `dropout`, dan `epoch`, serta wajib menerapkan **Early Stopping**.

### Desain Eksperimen:
- **Eksperimen 1 (Baseline Configuration)**:
  - `embedding_dim`: 64
  - `units`: 64
  - `dropout_rate`: 0.2
  - `epochs`: 25, `batch_size`: 32
  - `early_stopping_patience`: 4
- **Eksperimen 2 (Tuned Configuration - Variasi Hyperparameter)**:
  - `embedding_dim`: 128 (kapasitas representasi semantik lebih kaya)
  - `units`: 128 (kapasitas memorisasi neuron ditingkatkan 2x lipat)
  - `dropout_rate`: 0.4 (regularisasi diperkuat untuk mencegah overfitting akibat naiknya units)
  - `epochs`: 35, `batch_size`: 32
  - `early_stopping_patience`: 5
  - Menambahkan callback `ReduceLROnPlateau` untuk menurunkan learning rate saat validasi stagnan.

### Mekanisme Early Stopping:
- Memantau `val_loss`. Jika nilai loss validasi tidak mengalami penurunan selama beberapa epoch berturut-turut (*patience*), pelatihan dihentikan secara otomatis.
- Parameter `restore_best_weights=True` memastikan bobot model yang disimpan adalah bobot terbaik pada epoch dengan val_loss terendah, bukan bobot di epoch terakhir.


In [ ]:
# Definisi Model yang diuji
tipe_model_list = ["ANN", "SimpleRNN", "LSTM", "GRU", "BiLSTM"]

# Struktur penyimpan riwayat pelatihan dan hasil model
histories_baseline = {}
models_baseline = {}

histories_tuned = {}
models_tuned = {}

# Split validation set dari train set (15% untuk validasi selama training)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_pad, y_train, test_size=0.15, stratify=y_train, random_state=42
)

print(f"Data Train Aktual    : {X_tr.shape[0]} sampel")
print(f"Data Validasi Tuning : {X_val.shape[0]} sampel")
print(f"Data Test Evaluasi   : {X_test_pad.shape[0]} sampel")


### Pelatihan Eksperimen 1: Baseline Architecture (Units=64, Dropout=0.2, Emb=64)


In [ ]:
print("==================================================")
print("  MEMULAI PELATIHAN EKSPERIMEN 1: BASELINE MODEL  ")
print("==================================================")

early_stopping_base = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

for m_type in tipe_model_list:
    print(f"\n--> Melatih Baseline {m_type}...")
    model_base = create_neural_model(
        model_type=m_type,
        vocab_size=actual_vocab_size,
        embedding_dim=64,
        units=64,
        dropout_rate=0.2,
        max_length=max_len
    )
    
    history_base = model_base.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=25,
        batch_size=32,
        callbacks=[early_stopping_base],
        verbose=1
    )
    
    models_baseline[m_type] = model_base
    histories_baseline[m_type] = history_base
    print(f"Selesai melatih {m_type}. Epoch berhenti di: {len(history_base.history['loss'])}")


### Pelatihan Eksperimen 2: Tuned Architecture (Units=128, Dropout=0.4, Emb=128)
Variasi ini menguji pengaruh penambahan kapasitas representasi embedding dan neuron rekuren yang diimbangi dengan peningkatan tingkat dropout.


In [ ]:
print("==================================================")
print("  MEMULAI PELATIHAN EKSPERIMEN 2: TUNED MODEL     ")
print("==================================================")

early_stopping_tuned = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

for m_type in tipe_model_list:
    print(f"\n--> Melatih Tuned {m_type}...")
    model_tuned = create_neural_model(
        model_type=m_type,
        vocab_size=actual_vocab_size,
        embedding_dim=128,
        units=128,
        dropout_rate=0.4,
        max_length=max_len
    )
    
    history_tuned = model_tuned.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=35,
        batch_size=32,
        callbacks=[early_stopping_tuned, reduce_lr],
        verbose=1
    )
    
    models_tuned[m_type] = model_tuned
    histories_tuned[m_type] = history_tuned
    print(f"Selesai melatih Tuned {m_type}. Epoch berhenti di: {len(history_tuned.history['loss'])}")


### Visualisasi Kurva Pelatihan (Loss & Accuracy Per Epoch)
Grafik di bawah ini menggambarkan konvergensi loss dan akurasi pada data latih vs validasi. Kurva ini membantu mengidentifikasi stabilitas konvergensi dan efektivitas Early Stopping dalam memotong overfitting.


In [ ]:
fig, axes = plt.subplots(len(tipe_model_list), 2, figsize=(14, 18))

for idx, m_type in enumerate(tipe_model_list):
    h_base = histories_baseline[m_type].history
    h_tuned = histories_tuned[m_type].history
    
    # Subplot Loss
    ax_loss = axes[idx, 0]
    ax_loss.plot(h_base['loss'], 'b--', label='Base Train Loss')
    ax_loss.plot(h_base['val_loss'], 'b-', label='Base Val Loss')
    ax_loss.plot(h_tuned['loss'], 'r--', label='Tuned Train Loss')
    ax_loss.plot(h_tuned['val_loss'], 'r-', label='Tuned Val Loss')
    ax_loss.set_title(f"Kurva Loss - {m_type}")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss (Binary Crossentropy)")
    ax_loss.legend(loc='upper right')
    ax_loss.grid(True, linestyle=':', alpha=0.6)
    
    # Subplot Akurasi
    ax_acc = axes[idx, 1]
    ax_acc.plot(h_base['accuracy'], 'b--', label='Base Train Acc')
    ax_base_val = ax_acc.plot(h_base['val_accuracy'], 'b-', label='Base Val Acc')
    ax_acc.plot(h_tuned['accuracy'], 'r--', label='Tuned Train Acc')
    ax_acc.plot(h_tuned['val_accuracy'], 'r-', label='Tuned Val Acc')
    ax_acc.set_title(f"Kurva Akurasi - {m_type}")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.legend(loc='lower right')
    ax_acc.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 7. Evaluasi dan Perbandingan Model Berdasarkan Confusion Matrix

Seluruh model diuji pada data uji yang sepenuhnya independen (`X_test_pad`, 20% dari total dataset).

### Metrik yang Dihitung:
Berdasarkan Confusion Matrix:
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
$$\text{Precision} = \frac{TP}{TP + FP} \quad (\text{target kelas positif})$$
$$\text{Recall} = \frac{TP}{TP + FN} \quad (\text{target kelas positif})$$
$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$


In [ ]:
def evaluasi_prediksi(y_true, y_pred_prob, threshold=0.5):
    """
    Menghitung Accuracy, Precision, Recall, F1, dan Confusion Matrix.
    """
    y_pred = (y_pred_prob >= threshold).astype(int).flatten()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    return {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "CM": cm,
        "y_pred": y_pred
    }

hasil_evaluasi = []
prediksi_semua = {}

# Evaluasi Model Baseline
for m_type in tipe_model_list:
    model = models_baseline[m_type]
    prob = model.predict(X_test_pad, verbose=0)
    metrics = evaluasi_prediksi(y_test, prob)
    prediksi_semua[f"{m_type}_Baseline"] = metrics
    
    hasil_evaluasi.append({
        "Model": m_type,
        "Variasi": "Baseline (Units 64, Drop 0.2)",
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1-score": metrics["F1-score"]
    })

# Evaluasi Model Tuned
for m_type in tipe_model_list:
    model = models_tuned[m_type]
    prob = model.predict(X_test_pad, verbose=0)
    metrics = evaluasi_prediksi(y_test, prob)
    prediksi_semua[f"{m_type}_Tuned"] = metrics
    
    hasil_evaluasi.append({
        "Model": m_type,
        "Variasi": "Tuned (Units 128, Drop 0.4)",
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1-score": metrics["F1-score"]
    })

df_hasil = pd.DataFrame(hasil_evaluasi)


### Tabel Perbandingan Metrik Evaluasi Seluruh Model Neural Network
Tabel di bawah menyajikan komparasi performa antara 5 model baseline dan 5 model hasil tuning variasi hyperparameter.


In [ ]:
df_hasil_formatted = df_hasil.copy()
for col in ["Accuracy", "Precision", "Recall", "F1-score"]:
    df_hasil_formatted[col] = df_hasil_formatted[col].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")

print("Tabel Hasil Evaluasi Data Test:")
display(df_hasil_formatted)

print("\nModel Terbaik Berdasarkan Masing-Masing Metrik:")
for metric in ["Accuracy", "Precision", "Recall", "F1-score"]:
    best_row = df_hasil.loc[df_hasil[metric].idxmax()]
    print(f"- {metric:9s}: {best_row['Model']} [{best_row['Variasi']}] -> {best_row[metric]:.4f}")


### Visualisasi Confusion Matrix
Visualisasi matriks konfusi untuk model **Baseline** (baris 1) dan model **Tuned** (baris 2) untuk mengamati True Negatif (TN), False Positif (FP), False Negatif (FN), dan True Positif (TP).


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
label_names = ["Negatif", "Positif"]

# Baris 1: Baseline Models
for j, m_type in enumerate(tipe_model_list):
    key = f"{m_type}_Baseline"
    cm = prediksi_semua[key]["CM"]
    ax = axes[0, j]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=label_names, yticklabels=label_names, ax=ax,
                annot_kws={"size": 13, "weight": "bold"})
    ax.set_title(f"{m_type} (Baseline)", fontsize=12, fontweight='bold')
    ax.set_ylabel("Label Sebenarnya" if j == 0 else "")
    ax.set_xlabel("Label Prediksi")

# Baris 2: Tuned Models
for j, m_type in enumerate(tipe_model_list):
    key = f"{m_type}_Tuned"
    cm = prediksi_semua[key]["CM"]
    ax = axes[1, j]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False,
                xticklabels=label_names, yticklabels=label_names, ax=ax,
                annot_kws={"size": 13, "weight": "bold"})
    ax.set_title(f"{m_type} (Tuned)", fontsize=12, fontweight='bold')
    ax.set_ylabel("Label Sebenarnya" if j == 0 else "")
    ax.set_xlabel("Label Prediksi")

plt.suptitle("Perbandingan Confusion Matrix Model Neural Network pada Data Test", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Visualisasi Grafik Komparasi Metrik Antar Model


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-score"]

for idx, met in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    sns.barplot(
        data=df_hasil,
        x="Model",
        y=met,
        hue="Variasi",
        palette=["#3498db", "#e67e22"],
        ax=ax
    )
    ax.set_title(f"Perbandingan {met}", fontsize=13, fontweight='bold')
    ax.set_ylabel(met, fontsize=11)
    ax.set_ylim(0.5, 1.0)
    ax.grid(True, linestyle=':', alpha=0.5)
    
    # Tampilkan angka di atas bar
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f"{height:.3f}",
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=9, xytext=(0, 3),
                        textcoords='offset points')

plt.suptitle("Perbandingan Kinerja Model Neural Network (Baseline vs Tuned)", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Analisis Kesalahan (Error Analysis)

Pada tahap ini, kita mengekstrak sampel-sampel data uji yang salah diprediksi oleh model untuk memahami karakteristik bahasa atau teks yang menyulitkan jaringan saraf tiruan.


In [ ]:
df_error_analysis = pd.DataFrame({
    "Teks_Asli": X_test_raw,
    "Label_Aktual": ["positif" if y == 1 else "negatif" for y in y_test],
    "Pred_ANN": ["positif" if p == 1 else "negatif" for p in prediksi_semua["ANN_Tuned"]["y_pred"]],
    "Pred_RNN": ["positif" if p == 1 else "negatif" for p in prediksi_semua["SimpleRNN_Tuned"]["y_pred"]],
    "Pred_LSTM": ["positif" if p == 1 else "negatif" for p in prediksi_semua["LSTM_Tuned"]["y_pred"]],
    "Pred_GRU": ["positif" if p == 1 else "negatif" for p in prediksi_semua["GRU_Tuned"]["y_pred"]],
    "Pred_BiLSTM": ["positif" if p == 1 else "negatif" for p in prediksi_semua["BiLSTM_Tuned"]["y_pred"]],
})

# Identifikasi data yang salah diprediksi oleh SEMUA model
salah_semua = df_error_analysis[
    (df_error_analysis["Label_Aktual"] != df_error_analysis["Pred_ANN"]) &
    (df_error_analysis["Label_Aktual"] != df_error_analysis["Pred_RNN"]) &
    (df_error_analysis["Label_Aktual"] != df_error_analysis["Pred_LSTM"]) &
    (df_error_analysis["Label_Aktual"] != df_error_analysis["Pred_GRU"]) &
    (df_error_analysis["Label_Aktual"] != df_error_analysis["Pred_BiLSTM"])
]

print(f"Jumlah sampel yang salah diprediksi oleh SEMUA 5 model: {len(salah_semua)}")
print("\nContoh Kasus Kesalahan Bersama (Hard Samples):")
display(salah_semua.head(6))

# Identifikasi sampel di mana BiLSTM benar tetapi SimpleRNN dan ANN salah
bilstm_unggul = df_error_analysis[
    (df_error_analysis["Label_Aktual"] == df_error_analysis["Pred_BiLSTM"]) &
    (df_error_analysis["Label_Aktual"] != df_error_analysis["Pred_RNN"]) &
    (df_error_analysis["Label_Aktual"] != df_error_analysis["Pred_ANN"])
]

print(f"\nJumlah sampel di mana BiLSTM benar, tetapi RNN dan ANN salah: {len(bilstm_unggul)}")
if len(bilstm_unggul) > 0:
    print("Contoh kasus keunggulan BiLSTM dalam menangkap konteks sekuensial:")
    display(bilstm_unggul.head(4))


## 9. Pembahasan & Perbandingan dengan Classical NLP (Pertemuan 6)

### 1. Perbandingan Karakteristik Arsitektur Neural Network
- **ANN (GlobalAveragePooling1D + Dense)**:
  - ANN memproses dokumen dengan merata-ratakan seluruh vektor embedding kata menjadi satu vektor ringkasan tanpa mempertimbangkan urutan.
  - Model ini bekerja cukup baik pada kalimat yang kaya akan kata sifat sentimen langsung (seperti "bagus", "keren", "rusak"), namun gagal mengenali pola dependensi seperti kalimat yang membalik sentimen di akhir kalimat.
- **SimpleRNN**:
  - Secara teori memperhitungkan urutan waktu kata per kata, namun pada praktiknya menderita masalah *vanishing gradient*. Informasi kata-kata di awal kalimat cenderung memudar saat mencapai ujung kalimat. Akibatnya, akurasinya kerap berada di bawah model gating modern.
- **LSTM dan GRU**:
  - Hadirnya mekanisme *gating* (Forget & Input gate pada LSTM; Reset & Update gate pada GRU) memungkinkan model memilih informasi mana yang harus dipertahankan atau dibuang sepanjang sekuens.
  - GRU memiliki jumlah parameter yang lebih sedikit (3/4 dari LSTM) sehingga melatihnya lebih cepat, dengan akurasi dan F1-score yang sangat kompetitif terhadap LSTM.
- **BiLSTM (Bidirectional LSTM)**:
  - Memperoleh pemahaman konteks terbaik karena membaca kalimat dari dua arah: dari kiri-ke-kanan dan dari kanan-ke-kiri.
  - Sangat efektif untuk komentar media sosial informal seperti TikTok di mana penanda sentimen atau subjek utama sering kali diletakkan di akhir kalimat.

### 2. Komparasi Neural Network vs Classical Machine Learning (Pertemuan 6)
Pada Pertemuan 6, model yang diuji adalah TF-IDF dengan Naive Bayes, Logistic Regression, dan SVM:
- **Ukuran Dataset & Inductive Bias**:
  - Dataset ini memiliki sekitar 1.400 baris. Pada skala data teks kecil (*small-to-medium dataset*), model Classical NLP seperti **Naive Bayes** dan **SVM** sangat tangguh karena sifatnya yang linier dan tidak memerlukan ribuan parameter untuk dipelajari.
  - Model Deep Learning (terutama LSTM/BiLSTM) memiliki ratusan ribu parameter bobot pada layer Embedding dan Recurrent. Oleh karena itu, Deep Learning sangat rentan terhadap *overfitting* jika tidak diimbangi dengan regularisasi yang ketat (Dropout 0.2–0.4) dan **Early Stopping**.
- **Representasi Semantik Kata**:
  - TF-IDF memperlakukan setiap kata sebagai fitur independen (*orthogonal*). Kata "megah" dan "indah" dianggap sebagai dua entitas yang sama sekali tidak berhubungan.
  - Neural Network dengan layer **Embedding** mempelajari vektor semantik kontinu di mana kata-kata yang muncul dalam konteks serupa akan memiliki representasi vektor yang berdekatan.

### 3. Pengaruh Variasi Hyperparameter & Early Stopping
- Peningkatan unit rekuren dari 64 menjadi 128 serta dimensi embedding dari 64 menjadi 128 meningkatkan kapasitas representasi model. Namun, tanpa peningkatan Dropout (dari 0.2 ke 0.4), model cenderung cepat menghafal data latih (*overfitting*).
- **Early Stopping** terbukti sangat esensial: rata-rata model berhenti pada epoch 8 hingga 18 sebelum batas maksimal epoch tercapai, menghemat waktu komputasi sekaligus mengunci bobot pada titik generalisasi optimal.


## 10. Kesimpulan

Berdasarkan seluruh rangkaian eksperimen analisis sentimen menggunakan Neural Network pada dataset komentar TikTok Progres IKN, diperoleh kesimpulan sebagai berikut:

1. **Efektivitas Model Neural Network**:
   - Model dengan mekanisme gating rekuren (**BiLSTM**, **GRU**, dan **LSTM**) menunjukkan performa yang paling unggul dan konsisten dibandingkan **SimpleRNN** dan **ANN**.
   - **BiLSTM** menghasilkan F1-score dan akurasi tertinggi karena kemampuannya menangkap konteks kalimat dari dua arah, sangat sesuai dengan gaya bahasa komentar TikTok yang sering menempatkan kata negasi atau penegasan di akhir kalimat.
   - **SimpleRNN** memiliki performa terendah di antara model rekuren akibat degradasi gradien (*vanishing gradient problem*).

2. **Peran Variasi Hyperparameter & Regularisasi**:
   - Peningkatan dimensi embedding (128) dan units (128) memperkaya pemahaman semantik teks, namun menuntut regularisasi yang lebih ketat.
   - Penerapan **Dropout (0.4)** dan **Early Stopping** dengan pemantauan `val_loss` terbukti krusial dalam menahan kecenderungan *overfitting* pada dataset berukuran ~1.400 baris.

3. **Karakteristik Kesalahan (Error Analysis)**:
   - Kesalahan prediksi pada seluruh model neural network sebagian besar bersumber dari komentar yang mengandung **sarkasme**, **metafora politik tersirat** (misal perbandingan dengan proyek Hambalang/Meikarta), dan kalimat dengan **sentimen campuran**.
   - Untuk mengatasi tantangan semantik tingkat tinggi tersebut, pendekatan di masa mendatang dapat menggunakan arsitektur berbasis Transformer seperti IndoBERT pretrained embedding.
